In [1]:

# dependencies
!pip install -q sentence-transformers xgboost lightgbm scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

try:
    from lightgbm import LGBMClassifier
    lightgbm_available = True
except ImportError:
    lightgbm_available = False

from sentence_transformers import SentenceTransformer

# Clean function
def clean_dataframe(df):
    df['description'] = df['description'].fillna('unknown').str.lower().str.strip()
    df['merchantName'] = df['merchantName'].fillna('unknown').str.lower().str.strip()
    df['subcategory'] = df['subcategory'].fillna('unknown').str.lower().str.strip()
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce').fillna(0.0)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    if 'merchantCode' in df.columns:
        df['merchantCode'] = df['merchantCode'].fillna(-1)
    if 'type' in df.columns:
        df['type'] = df['type'].astype(str).str.lower().str.strip()
    if 'category' in df.columns:
        df['category'] = df['category'].astype(str).str.lower().str.strip()
    return df

# Split and Balance Function
def split_and_balance(df, text_column="description", label_column="subcategory", min_samples=300):
    from collections import Counter

    label_encoder = LabelEncoder()
    df[label_column] = label_encoder.fit_transform(df[label_column])


    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df[label_column]
    )
    print(f"Original dataset: {len(df)} samples")
    print(f"Training set: {len(train_df)} samples")
    print(f"Test set: {len(test_df)} samples (stratified)")

    print("\nOriginal training class distribution:\n", Counter(train_df[label_column]))

    balanced_train_data = []
    for subcat, group in train_df.groupby(label_column):
        if len(group) < min_samples:
            sampled = group.sample(n=min_samples, replace=True, random_state=42)
        else:
            sampled = group
        balanced_train_data.append(sampled)

    balanced_train_df = pd.concat(balanced_train_data).reset_index(drop=True)

    print("\nBalanced training class distribution:\n", Counter(balanced_train_df[label_column]))

    X_train = balanced_train_df[text_column]
    y_train = balanced_train_df[label_column]
    X_test = test_df[text_column]
    y_test = test_df[label_column]

    return X_train, X_test, y_train, y_test, label_encoder

# Prepare Data Function
def prepare_data_balanced(df, text_column="description", label_column="subcategory", min_samples=300):
    # Split & balance
    X_train, X_test, y_train, y_test, label_encoder = split_and_balance(
        df, text_column, label_column, min_samples
    )


    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    print("\nEncoding training data...")
    X_train_embeddings = encoder.encode(X_train.tolist(), show_progress_bar=True)
    print("Encoding test data...")
    X_test_embeddings = encoder.encode(X_test.tolist(), show_progress_bar=True)

    return X_train_embeddings, X_test_embeddings, y_train, y_test, encoder, label_encoder

# Model Comparison Function
def compare_models(X_train, X_test, y_train, y_test, label_encoder):
    results = []
    models = {
        "XGBoost": xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=len(label_encoder.classes_),
            eval_metric='mlogloss',
            max_depth=6,
            n_estimators=300,
            learning_rate=0.1,
            use_label_encoder=False,
            verbosity=0
        ),
        "Logistic Regression": LogisticRegression(
            max_iter=1000, multi_class='multinomial', solver='lbfgs'
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, max_depth=10, random_state=42
        )
    }
    if lightgbm_available:
        models["LightGBM"] = LGBMClassifier(n_estimators=300, max_depth=10, random_state=42)

    best_test_f1 = -float('inf')
    best_model_name = None
    best_model = None

    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        train_f1 = f1_score(y_train, y_train_pred, average="weighted")
        test_f1 = f1_score(y_test, y_test_pred, average="weighted")

        results.append({
            "Model": name,
            "Train F1": train_f1,
            "Test F1": test_f1
        })

        if test_f1 > best_test_f1:
            best_test_f1 = test_f1
            best_model_name = name
            best_model = model

    results_df = pd.DataFrame(results)
    print("\n==============================")
    print(results_df.sort_values("Test F1", ascending=False).to_string(index=False))
    print("==============================\n")
    print(f"Best Model: {best_model_name} (Test F1: {best_test_f1:.2f})")

    return best_model_name, best_model

# Evaluate Function
def evaluate_model(model, X_train, y_train, X_test, y_test, label_encoder):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    train_f1 = f1_score(y_train, y_train_pred, average="weighted")
    test_f1 = f1_score(y_test, y_test_pred, average="weighted")

    print("\nTraining Accuracy: {:.2f}%".format(train_acc * 100))
    print("Test Accuracy: {:.2f}%".format(test_acc * 100))

    print("\nTraining F1 Score: {:.2f}".format(train_f1))
    print("Test F1 Score: {:.2f}".format(test_f1))

    if (train_acc - test_acc > 0.05) or (train_f1 - test_f1 > 0.05):
        print("\ Warning: Possible overfitting detected!")
    else:
        print("\ No major overfitting detected.")

    print("\nClassification Report on Test Set:")
    print(classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))

    conf = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(14, 12))
    sns.heatmap(conf, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title("Confusion Matrix on Test Set")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.3 MB/s eta 0:00:00


In [ ]:

df = pd.read_csv("/content/30000_transactions.csv")
df = clean_dataframe(df)
X_train_emb, X_test_emb, y_train, y_test, encoder, label_encoder = prepare_data_balanced(df)
best_model_name, best_model = compare_models(X_train_emb, X_test_emb, y_train, y_test, label_encoder)
evaluate_model(best_model, X_train_emb, y_train, X_test_emb, y_test, label_encoder)


In [1]:
# Hyperparameter Tuning
def tune_hyperparameters(model, X_train, y_train, n_iter=25, cv=3, random_state=42):
    from sklearn.model_selection import RandomizedSearchCV


    if isinstance(model, xgb.XGBClassifier):
        param_grid = {
            "max_depth": [3, 5, 7],
            "learning_rate": [0.01, 0.05, 0.1],
            "n_estimators": [100, 200, 300],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "reg_alpha": [0, 0.01, 0.1, 1],
            "reg_lambda": [0.1, 1, 5, 10]
        }
    elif isinstance(model, RandomForestClassifier):
        param_grid = {
            "n_estimators": [100, 200, 300],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2"]
        }
    elif isinstance(model, LogisticRegression):
        param_grid = {
            "C": [0.01, 0.1, 1, 10],
            "solver": ["lbfgs", "saga"],
            "penalty": ["l2"]
        }
    elif lightgbm_available and isinstance(model, LGBMClassifier):
        param_grid = {
            "num_leaves": [31, 50, 70],
            "learning_rate": [0.01, 0.05, 0.1],
            "n_estimators": [100, 200, 300],
            "max_depth": [-1, 10, 20],
            "reg_alpha": [0, 0.01, 0.1, 1],
            "reg_lambda": [0.1, 1, 5, 10]
        }
    else:
        raise ValueError("Unsupported model type for tuning!")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=n_iter,
        scoring="f1_weighted",
        cv=cv,
        verbose=2,
        random_state=random_state,
        n_jobs=-1
    )

    print("\ Starting hyperparameter tuning...")
    search.fit(X_train, y_train)

    print("\ Best Hyperparameters Found:")
    print(search.best_params_)
    print(f"Best Cross-Validated F1 Score: {search.best_score_:.2f}")

    best_model = search.best_estimator_
    return best_model


In [ ]:

best_model_tuned = tune_hyperparameters(best_model, X_train_emb, y_train, n_iter=30, cv=3)
evaluate_model(best_model_tuned, X_train_emb, y_train, X_test_emb, y_test, label_encoder)


In [ ]:
# Save model function
def save_pipeline(model, encoder, label_encoder, save_dir="saved_pipeline"):
    os.makedirs(save_dir, exist_ok=True)

    # Save the trained model
    model_path = os.path.join(save_dir, "trained_model.joblib")
    joblib.dump(model, model_path)
    print(f" Saved model to {model_path}")

    # Save the sentence transformer encoder
    encoder_path = os.path.join(save_dir, "sentence_transformer")
    encoder.save(encoder_path)
    print(f" Saved sentence transformer encoder to {encoder_path}")

    # Save the label encoder
    label_encoder_path = os.path.join(save_dir, "label_encoder.joblib")
    joblib.dump(label_encoder, label_encoder_path)
    print(f"Saved label encoder to {label_encoder_path}")
